# Collection A Processing: Split by Method (TMT, iTRAQ, Label-Free)

This notebook processes **Collection A** cohorts by grouping them into 3 distinct categories based on experimental method:
1.  **TMT** (CPTAC/ICPC/CNHPP)
2.  **iTRAQ** (Historical CPTAC)
3.  **Label-Free** (Other matched cohorts)

For each group, we:
- Load the cohort datasets.
- Concatenate them into a unified matrix.
- Filter Features (>50% missing) and Samples (>50% missing).
- Save separate integrated files for Protein and RNA.

In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "1. process_collection_a"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
from pathlib import Path
import sys

# Make the adjacent src/ package importable when this notebook is run from the notebooks folder.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import src.utils_val as uv

In [ ]:
import os
import glob
import pandas as pd
from tqdm import tqdm
import warnings


In [ ]:

warnings.simplefilter(action='ignore', category=FutureWarning)

DATA_DIR = r"../data/Collection_A"
OUTPUT_DIR = "../data/processed"
if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

# --- DEFINING COHORT GROUPS ---
COHORT_GROUPS = {
    "TMT": [
        # CPTAC / ICPC / CNHPP
        "colon_2019", "renal_2019", "Endomet_val", "breast2020", "pedbrain_2020",
        "lung_2020_smoke", "lung_val", "lung2021", "headneck_2021", "pancrea2021",
        "brain_2021", "liver_val"
    ],
    "iTRAQ": [
        # Historical CPTAC
        "breast_2016", "ovary_2016", "stomach_2019"
    ],
    "LabelFree": [
        # Other matched cohorts
        "brain", "colon", "prostate", "liver", "lung"
    ]
}

# Initialize storage
group_data = {"TMT": {"prot": [], "rna": []}, "iTRAQ": {"prot": [], "rna": []}, "LabelFree": {"prot": [], "rna": []}}


## 1. Load Data into Groups

In [ ]:
subdirs = [f.path for f in os.scandir(DATA_DIR) if f.is_dir()]
print(f"Found {len(subdirs)} subdirectories in {DATA_DIR}")

for subdir in tqdm(subdirs, desc="Processing Cohorts"):
    cohort_name = os.path.basename(subdir)
    
    # Determine Group
    assigned_group = None
    for group_name, cohort_list in COHORT_GROUPS.items():
        if cohort_name in cohort_list:
            assigned_group = group_name
            break
    
    if not assigned_group:
        print(f"[WARNING] Cohort '{cohort_name}' not in defined groups. Skipping.")
        continue

    # Find Files
    prot_files = glob.glob(os.path.join(subdir, "protein_matrix_*.txt"))
    rna_files = glob.glob(os.path.join(subdir, "RNA_matrix_*.txt"))
    
    if prot_files and rna_files:
        try:
            df_p = pd.read_csv(prot_files[0], sep="\t", index_col=0)
            df_r = pd.read_csv(rna_files[0], sep="\t", index_col=0)
            
            # Align Samples
            common_samples = df_p.columns.intersection(df_r.columns)
            
            if len(common_samples) > 0:
                df_p = df_p[common_samples]
                df_r = df_r[common_samples]

                # Rename Samples
                new_names = [f"{cohort_name.upper()}_{i+1}" for i in range(len(common_samples))]
                df_p.columns = new_names
                df_r.columns = new_names
                
                # Store
                group_data[assigned_group]["prot"].append(df_p)
                group_data[assigned_group]["rna"].append(df_r)
        except Exception as e:
            print(f"  -> ERROR in {cohort_name}: {e}")

## 2. Concatenate, Filter, and Save per Group

In [ ]:
from src.utils_val import filter_nans

for group_name, dfs in group_data.items():
    print(f"\n=== Processing Group: {group_name} ===")
    
    # Concatenate
    full_protein = pd.concat(dfs["prot"], axis=1, join='outer', sort=True) if dfs["prot"] else pd.DataFrame()
    full_rna = pd.concat(dfs["rna"], axis=1, join='outer', sort=True) if dfs["rna"] else pd.DataFrame()

    # --- MAPPING RNA IDs to SYMBOLS ---
    MAPPING_FILE = os.path.abspath("../data/Collection_A/brain/rows_metadata.csv")
    if os.path.exists(MAPPING_FILE) and not full_rna.empty:
        print(" Mapping RNA Ensembl IDs to Gene Symbols...")
        try:
            map_df = pd.read_csv(MAPPING_FILE)
            map_df = map_df.dropna(subset=['ensembl_gene_id', 'gene_symbol'])
            id2sym = dict(zip(map_df['ensembl_gene_id'], map_df['gene_symbol']))
            
            # Map index
            full_rna.index = full_rna.index.map(lambda x: id2sym.get(x, x))
            
            # Handle duplicates (averaging)
            if full_rna.index.duplicated().any():
                print(f"   -> Found {full_rna.index.duplicated().sum()} duplicate symbols. Averaging...")
                full_rna = full_rna.groupby(level=0).mean()
                
            print("   -> Mapping complete.")
        except Exception as e:
            print(f"   -> Mapping Error: {e}")
    # ----------------------------------
    
    
    if full_protein.empty: 
        print(" [WARNING] No Protein data found.")
        continue
        
    print(f" Initial Shapes -> Protein: {full_protein.shape}, RNA: {full_rna.shape}")

    # Filter
    print(" Filtering Protein...")
    full_protein = filter_nans(full_protein)
    print(" Filtering RNA...")
    full_rna = filter_nans(full_rna)
    
    # Transpose (Samples x Features)
    integrated_protein = full_protein.T
    integrated_rna = full_rna.T
    
    # Remove duplicates
    integrated_protein = integrated_protein.loc[~integrated_protein.index.duplicated(keep='first')]
    integrated_rna = integrated_rna.loc[~integrated_rna.index.duplicated(keep='first')]
    
    print(f" Final Shapes   -> Protein: {integrated_protein.shape}, RNA: {integrated_rna.shape}")

    # --- VIEW HEADS ---
    print(f"Top 5 rows of {group_name} Protein:")
    display(integrated_protein.head())
    print(f"Top 5 rows of {group_name} RNA:")
    display(integrated_rna.head())
    # ------------------
    

    # Save
    p_out = os.path.join(OUTPUT_DIR, f"Collection_A_{group_name}_Protein.csv")
    r_out = os.path.join(OUTPUT_DIR, f"Collection_A_{group_name}_RNA.csv")
    
    integrated_protein.to_csv(p_out)
    integrated_rna.to_csv(r_out)
    print(f" Saved to {p_out} and {r_out}")